## Preprocessing TWAS-Tissue and TWAS-Cell-Type results for downstream plotting

**Author:** Michelle Franc Ragsac (mragsac@ucsd.edu)

In this notebook, I'll be preprocessing Nan's results after running FUSION-TWAS across 39 GTEx bulk tissues and 17 OneK1K pseudobulked cell types derived from single-cell sequencing of PBMCs. Currently, we have all of the data within multiple tables (`.dat`); the goal will be to consolidate the TWAS results within bulk tissue and pseudobulked cell types as two separate files to work with.

In [1]:
import glob
import pandas as pd
import numpy as np

## Import TWAS results tables

In [2]:
fns_twas_tissue = glob.glob("data/TWAS_tissue_results/twas_*_chr*.dat")
fns_twas_tissue += glob.glob("data/TWAS_tissue_results/twas_*_chr6.dat.MHC")

fns_twas_celltype = glob.glob("data/TWAS_celltype_results/TWAS_metaGWAS_*_chr*.dat")
fns_twas_celltype += glob.glob("data/TWAS_celltype_results/TWAS_metaGWAS_*_chr6.dat.MHC")

print(f"Number of Files to Process in data/TWAS_tissue_results/   : {len(fns_twas_tissue)}")
print(f"Number of Files to Process in data/TWAS_celltype_results/ : {len(fns_twas_celltype)}")

fns_twas_tissue = sorted(fns_twas_tissue)
fns_twas_celltype = sorted(fns_twas_celltype)

Number of Files to Process in data/TWAS_tissue_results/   : 897
Number of Files to Process in data/TWAS_celltype_results/ : 391


## Concatenate TWAS results tables

In [3]:
data = []
for fn in fns_twas_tissue:
    data.append(pd.read_table(fn))
df_twas_tissue = pd.concat(data)
df_twas_tissue = df_twas_tissue.sort_values(["PANEL","CHR","P1"])

data = []
for fn in fns_twas_celltype:
    data.append(pd.read_table(fn))
df_twas_celltype = pd.concat(data)
df_twas_celltype = df_twas_celltype.sort_values(["PANEL","CHR","P1"])

# We know that there are a few columns that don't have data types that we would expect, so replace the typing
df_twas_tissue["EQTL.ID"] = df_twas_tissue["EQTL.ID"].apply(lambda r: str(r).strip())
df_twas_celltype["EQTL.ID"] = df_twas_celltype["EQTL.ID"].apply(lambda r: str(r).strip())

columns = ["BEST.GWAS.Z", "EQTL.R2", "EQTL.Z", "EQTL.GWAS.Z", "TWAS.Z", "TWAS.P"]
for column in columns:
    df_twas_tissue[f"{column}.CLEANED"] = pd.to_numeric(
        df_twas_tissue[column].apply(lambda r: str(r).strip()).replace("NA",np.nan).replace("nan",np.nan))
    df_twas_celltype[f"{column}.CLEANED"] = pd.to_numeric(
        df_twas_celltype[column].apply(lambda r: str(r).strip()).replace("NA",np.nan).replace("nan",np.nan))

# Replace the columns that we modified for typing
df_twas_tissue = df_twas_tissue.drop(columns=columns).rename(columns={f"{c}.CLEANED":c for c in columns})
df_twas_celltype = df_twas_celltype.drop(columns=columns).rename(columns={f"{c}.CLEANED":c for c in columns})

df_twas_tissue = df_twas_tissue.reset_index(drop=True)
df_twas_celltype = df_twas_celltype.reset_index(drop=True)

print(f"Dimensions of the TWAS Tissue Table Results : {df_twas_tissue.shape} (rows, columns)")
print(f"Dimensions of the TWAS Cell-type Table Results : {df_twas_celltype.shape} (rows, columns)")

Dimensions of the TWAS Tissue Table Results : (226122, 20) (rows, columns)
Dimensions of the TWAS Cell-type Table Results : (19055, 20) (rows, columns)


## Clean up Tissue and Cell Type Names

In [4]:
tissues = {
    'adipose_subcutaneous': 'adipose (subcutaneous)',
    'adipose_visceral_omentum': 'adipose (visceral omentum)',
    'adrenal_gland': 'adrenal gland',
    'artery_aorta': 'artery (aorta)',
    'artery_coronary': 'artery (coronary)',
    'artery_tibial': 'artery (tibial)',
    'brain_basalganglia': 'brain (basal ganglia)',
    'brain_cerebellum': 'brain (cerebellum)',
    'brain_cortex': 'brain (cortex)',
    'brain_limbic': 'brain (limbic)',
    'brain_spinal_cord_cervical_c-1': 'brain (spinal cord - cervical C1)',
    'brain_substantia_nigra': 'brain (substantia nigra)',
    'breast_mammary_tissue': 'breast (mammary tissue)',
    'cells_cultured_fibroblasts': 'cells (cultured fibroblasts)',
    'cells_ebv-transformed_lymphocytes': 'cells (EBV-transformed lymphocytes)',
    'colon_sigmoid': 'colon (sigmoid)',
    'colon_transverse': 'colon (transverse)',
    'esophagus_mucosa': 'esophagus mucosa',
    'esophagus_muscularis': 'esophagus muscularis',
    'heart_atrial_appendage': 'heart (atrial appendage)',
    'heart_left_ventricle': 'heart (left ventricle)',
    'liver': 'liver',
    'lung': 'lung',
    'minor_salivary_gland': 'minor salivary glands',
    'muscle_skeletal': 'muscle (skeletal)',
    'nerve_tibial': 'nerve (tibial)',
    'ovary': 'ovary',
    'pancreas': 'pancreas',
    'pituitary': 'pituitary',
    'prostate': 'prostate',
    'skin_not_sun_exposed_suprapubic': 'skin (not sun exposed, suprapubic)',
    'skin_sun_exposed_lower_leg': 'skin (sun exposed, lower leg)',
    'spleen': 'spleen',
    'stomach': 'stomach',
    'testis': 'testis',
    'thyroid': 'thyroid',
    'uterus': 'uterus',
    'vagina': 'vagina',
    'whole_blood': 'whole blood'
}

celltypes = {
    'b_intermediate': 'intermediate B Cells',
    'b_memory': 'memory B Cells',
    'b_naive': 'naïve B cells',
    'cd14_mono': 'CD14+ monocytes',
    'cd16_mono': 'CD16+ monocytes',
    'cd4_ctl': 'CD4+ cytotoxic T lymphocytes (CTL)',
    'cd4_naive': 'CD4+ naïve T cells',
    'cd4_tcm': 'CD4+ central memory T (Tcm) cells',
    'cd4_tem': 'CD4+ effector memory T (Tem) cells',
    'cd8_naive': 'CD8+ naïve T cells',
    'cd8_tcm': 'CD8+ central memory T (Tcm) cells',
    'cd8_tem': 'CD8+ effector memory T (Tem) cells',
    'gdt': 'gdT',
    'mait': 'MAIT',
    'nk': 'natural killer (NK) cells',
    'nk_cd56bright': 'CD56bright natural killer (NK) cells',
    'treg': 'Treg'
}

# Add tissue and cell type information in a cleaner format for plotting purposes later
df_twas_tissue["PANEL.CLEANED"] = df_twas_tissue["PANEL"].apply(lambda r: tissues[r])
df_twas_celltype["PANEL.CLEANED"] = df_twas_celltype["PANEL"].apply(lambda r: celltypes[r])

## Export concatenated TWAS results tables for downstream plotting

In [5]:
df_twas_tissue.to_csv("data/twas_all-tissue-all-chr.tsv", sep="\t")
df_twas_celltype.to_csv("data/twas_all-celltype-all-chr.tsv", sep="\t")